In [15]:
# import zipfile
# import glob

# years = [2025]
# folder = "/home/rishi/ML Projects/Air Pollution/Datasets/epa_data"

# for f in glob.glob(f"{folder}/*.zip"):
#     if any(f.endswith(f"{year}.zip") for year in years):
#         with zipfile.ZipFile(f, "r") as z:
#             z.extractall("output_folder/")


In [16]:
import pandas as pd
import os

types=['hourly_42401',
 'hourly_88101',
 'hourly_44201',
 'hourly_42602',
 'hourly_42101',
 'hourly_81102',
]

In [17]:
def separate_and_filter(df, pol):
    df["Timestamp"] = pd.to_datetime(df["Date GMT"] + " " + df["Time GMT"], format="%Y-%m-%d %H:%M")
    s=df['Parameter Name'].iloc[0]+' ' +df['Units of Measure'].iloc[0]
    year=pd.to_datetime(df["Date Local"].iloc[0]).year
    df[s]=df['Sample Measurement']
    df["Key"]=df.apply(lambda x: (x['State Code'],x['County Code'], x['Site Num']), axis=1)
    df_f=df[['Key','Timestamp', 'Latitude', 'Longitude',s]]
    output_dir = "epa_data_by_site"
    os.makedirs(output_dir, exist_ok=True)

    for site_id, group_df in df_f.groupby("Key"):
        safe_name = f"site_{site_id[0]}_{site_id[1]}_{site_id[2]}_{pol}_{year}"
        safe_name = safe_name.replace("/", "_").replace(" ", "_")
        group_df.to_csv(os.path.join(output_dir, f"{safe_name}.csv"), index=False)
    print("Saved ", pol)

In [18]:
base = r"/home/rishi/ML Projects/Air Pollution/EPA/output_folder/"
year = 2025
output_dir = "epa_data_by_site"
os.makedirs("joined", exist_ok=True)


In [19]:
type_to_pol={
    'hourly_42401':'Sulphur Dioxide',
 'hourly_88101': 'PM2.5',
 'hourly_44201': 'Ozone',
 'hourly_42602': 'Nitrogen Dioxide',
 'hourly_42101':'Carbon Monoxide',
 'hourly_81102': 'PM10',
}
# Save per-site CSVs for each type
for type in types:
    df = pd.read_csv(fr"{base}{type}_{year}.csv")
    separate_and_filter(df, type_to_pol[type])


/tmp/ipykernel_1025/1702825594.py:11: DtypeWarning: Columns (0: Qualifier) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(fr"{base}{type}_{year}.csv")


Saved  Sulphur Dioxide


/tmp/ipykernel_1025/1702825594.py:11: DtypeWarning: Columns (0: Qualifier) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(fr"{base}{type}_{year}.csv")


Saved  PM2.5


/tmp/ipykernel_1025/1702825594.py:11: DtypeWarning: Columns (0: Qualifier) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(fr"{base}{type}_{year}.csv")


...

Saved  Nitrogen Dioxide


/tmp/ipykernel_1025/1702825594.py:11: DtypeWarning: Columns (0: Qualifier) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(fr"{base}{type}_{year}.csv")


Saved  Carbon Monoxide


/tmp/ipykernel_1025/1702825594.py:11: DtypeWarning: Columns (0: Qualifier) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(fr"{base}{type}_{year}.csv")


Saved  PM10


In [ ]:
# # Discover the actual column name for each pollutant type from any existing file
# pol_col_names = {}
# for t in types:
#     pol = type_to_pol[t].replace(" ", "_")
#     for f in os.listdir(output_dir):
#         if f.startswith("site_") and f.endswith(f"_{pol}_{year}.csv") and "_joined" not in f:
#             sample = pd.read_csv(os.path.join(output_dir, f), nrows=1)
#             extra = [c for c in sample.columns if c not in ["Key", "Timestamp", "Latitude", "Longitude"]]
#             if extra:
#                 pol_col_names[t] = extra[0]
#             break

# # Collect all site IDs
# site_ids = set()
# for t in types:
#     pol = type_to_pol[t].replace(" ", "_")
#     for f in os.listdir(output_dir):
#         prefix, suffix = "site_", f"_{pol}_{year}.csv"
#         if f.startswith(prefix) and f.endswith(suffix) and "_joined" not in f:
#             site_ids.add(f[len(prefix):-len(suffix)])

# join_on = ["Key", "Timestamp", "Latitude", "Longitude"]

# for site_id in site_ids:
#     type_dfs = []
#     for t in types:
#         pol = type_to_pol[t].replace(" ", "_")
#         fpath = os.path.join(output_dir, f"site_{site_id}_{pol}_{year}.csv")
#         if os.path.exists(fpath):
#             type_dfs.append(pd.read_csv(fpath))

#     if not type_dfs:
#         continue

#     merged = type_dfs[0]
#     for other in type_dfs[1:]:
#         dup_cols = [c for c in other.columns if c in merged.columns and c not in join_on]
#         merged = merged.merge(other.drop(columns=dup_cols), on=join_on, how="outer")

#     # Add NaN columns for any missing pollutants
#     for t, col_name in pol_col_names.items():
#         if col_name not in merged.columns:
#             merged[col_name] = float("nan")

#     out_path = os.path.join("joined", f"site_{site_id}_{year}_joined.csv")
#     merged.to_csv(out_path, index=False)
#     print(f"Saved joined: site_{site_id}_{year}_joined.csv ({len(merged)} rows)")